#  Introdução

Vamos fazer uma analise de sentimentos baseado em comentários de clientes ao comprarem determinados produtos em uma loja on-line. Dividiremos esses sentimentos em 'Positive', 'Neutral' e 'Negative'.


# Bibliografia

* [Documentação - Camadas do keras](https://www.tensorflow.org/api_docs/python/tf/keras/layers)


# Requirements

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
import nltk
import emoji
import spacy
import string
import tensorflow as tf
import datetime
from sklearn.utils import shuffle
from joblib import dump, load
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import CSVLogger, TensorBoard, EarlyStopping
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

# Reconhecendo o dataframe

In [2]:
# transformando e abrindo o arquivo em formato csv
df = pd.read_excel('DateToTestSentiment_20210817.xls.xls')

df.head(3)

,KEY,AP2,COLLECT_DATE,COMMENT_DATE,COMMENT_DISLIKE,COMMENT_ID,COMMENT_LIKE,COMMENT_TEXT,COMMENT_TITLE,COMPANY_NAME,COUNTRY,LAST_CRAWLER_DATE,PROD_BRAND,PROD_DESC,PROD_HREF,PRODUCT_TYPE,RATING_VALUE,SITE
0,aHR0cHM6Ly93d3cuY2FzYXNiYWhpYS5jb20uYnIvc21hcn...,SEDA,2021-06-18,16/06/21,0.0,9018105,0.0,Ótimo,NaN,CASAS BAHIA,BR,2021-06-18 15:40,SAMSUNG,Smartphone Samsung Galaxy A21s Branco 64GB Câm...,https://www.casasbahia.com.br/smartphone-samsu...,IM,5,https://www.casasbahia.com.br/
1,aHR0cHM6Ly93d3cuY2FzYXNiYWhpYS5jb20uYnIvc21hcn...,SEDA,2021-06-18,15/06/21,0.0,9011802,0.0,Ótimo,NaN,CASAS BAHIA,BR,2021-06-18 15:41,SAMSUNG,Smartphone Samsung Galaxy A21s Branco 64GB Câm...,https://www.casasbahia.com.br/smartphone-samsu...,IM,5,https://www.casasbahia.com.br/
2,aHR0cHM6Ly93d3cuY2FzYXNiYWhpYS5jb20uYnIvc21hcn...,SEDA,2021-06-18,15/06/21,0.0,9011259,0.0,Ótimo,NaN,CASAS BAHIA,BR,2021-06-18 15:41,SAMSUNG,Smartphone Samsung Galaxy A21s Branco 64GB Câm...,https://www.casasbahia.com.br/smartphone-samsu...,IM,5,https://www.casasbahia.com.br/


In [ ]:
# conferindo as colunas

df.shape, df.columns

In [ ]:
# dados nulos
df.isnull().sum()

In [3]:
# carrega o dataset apenas com as colunas desejadas
df = df[['COMMENT_TEXT','RATING_VALUE']]

df.head()

,COMMENT_TEXT,RATING_VALUE
0,Ótimo,5
1,Ótimo,5
2,Ótimo,5
3,Muito bom,5
4,Ótimo,5


In [4]:
# tira as linhas nulas
df = df.dropna().reset_index().drop(columns=['index'])

df.head()

,COMMENT_TEXT,RATING_VALUE
0,Ótimo,5
1,Ótimo,5
2,Ótimo,5
3,Muito bom,5
4,Ótimo,5


In [5]:
# verifica quantidade de nulos
df.isnull().sum()

COMMENT_TEXT    0
RATING_VALUE    0
dtype: int64

In [6]:
# faz value counts da variável rating_value
df['RATING_VALUE'].value_counts()

5       46880
4        7835
3        1811
5,0      1286
1         826
2         397
4,0       227
1,0       100
3,0        76
2,0        39
Name: RATING_VALUE, dtype: int64

In [7]:
# trata a coluna de rating para ficar apenas com dados float
for n, item in enumerate(df['RATING_VALUE']):
    if type(item) == str:
        df['RATING_VALUE'][n] = re.sub("[' ']","", df['RATING_VALUE'].iloc[n])
        df['RATING_VALUE'][n] = re.sub("[,]",".", df['RATING_VALUE'].iloc[n])

In [8]:
# transforma a coluna rating_value em float32
df['RATING_VALUE'] = df['RATING_VALUE'].astype('float32')

In [9]:
# faz value counts da variável rating_value
df['RATING_VALUE'].value_counts()

5.0    48166
4.0     8062
3.0     1887
1.0      926
2.0      436
Name: RATING_VALUE, dtype: int64

In [10]:
# separa a tabela em quantidade dados parecidas em cada label
dff = df.copy()

#
dff1 = dff.loc[dff['RATING_VALUE'] == 5, :]

#
dff2 = dff.loc[dff['RATING_VALUE'] != 5, :]

#
dff1['RATING_VALUE'].value_counts(), dff2['RATING_VALUE'].value_counts()

#
dff1 = dff1.sample(8062)

#
dff1.shape

(8062, 2)

In [11]:
# contrói a tabela final
df_final = dff1.append(dff2).reset_index().drop(columns=['index'])

#
df_final.shape

(19373, 2)

In [ ]:
df_final['RATING_VALUE'].value_counts()

In [12]:
# organiza os labels pelo valor da rating dado pelo cliente
dict_feeling = {2: 'Positive', 1: 'Negative', 0: 'Neutral'}


def sentimento(x):
    """
    Função que transforma a coluna passada como argumento
    na variável target.
    """
    
    if x == 5:
        return 2
    elif x == 4:
        return 0
    else:
        return 1

In [13]:
# contruindo a coluna nova com os sentimentos
df_final['sentimentos'] = df_final['RATING_VALUE'].apply(sentimento)

df_final.head()

,COMMENT_TEXT,RATING_VALUE,sentimentos
0,aparelho básico porem muito rápido entrega per...,5.0,2
1,otimo!,5.0,2
2,Muito bom,5.0,2
3,Muito Bom,5.0,2
4,Muito bom,5.0,2


In [ ]:
df_final['sentimentos'].value_counts()

# Tratamento de dados (funções)

Criando uma função para tratamento de datasets no contexto de analise de sentimentos. Vamos agora juntar todos os passos feitos acima em uma função, que poderemos usar em outras ocasiões de analise de sentimentos em outros datasets.

In [ ]:
def preprocess_data(data, columns,
                    label_column, null=True,
                    labels_ok=True, dict_labels=None,
                    labels_ready=True, labels_true=None):
    """ Função para tratamento de datasets para analise de sentimentos
    
    Args:
        data (pandas dataset) = Dataset 
        columns (list) = nome da coluna com texto para analise e da coluna com os labels
        null (Boolean) = define se vai limpar ou não os dados faltantes.
        label_columns (str)= nome da coluna com os label
        label_ok (Boolean) = define se a coluna de label está tratada ou não
        dict_label (dict) = define pelas chaves e valores as substituições vão ser feitas na coluna de labels.
        labels_ready (Boolean) = define se vamos usar os labels sem nenhuma alteração
        labels_true (dict) = dicionario com chaves sendo tuplas (com pelo menos dois elementos,
        mesmo que sejam elementos repetidos), com labels e valores sendo os respectivos 
        sentimentos (na ordem: positive, neutral e negative).
    """
    df = data[columns]
    
    if null:
        df = df.dropna().reset_index().drop(columns=['index'])
    
    if not labels_ok:
        for n, item in enumerate(df[label_column]):
            if type(item) == str:
                for key, value in dict_labels.items():
                    df[label_column][n] = re.sub(str(list(key)),value, df[label_column].iloc[n])
        df[label_column] = df[label_column].astype('float32')
                    
    if not labels_ready:
        def sentimento(x):
            if (x in [*labels_true.keys()][0]):
                return [*labels_true.values()][0]
            elif (x in [*labels_true.keys()][1]):
                return [*labels_true.values()][1]
            else:
                return [*labels_true.values()][2]
            
    df['sentimentos'] = df[label_column].apply(sentimento)    
    
    return df

# Teste na tabela original

Nossa tabela já está pré-processada, porém testaremos a função na tabela original como teste de sanidade da função criada.

In [ ]:
df2 = pd.read_excel('DateToTestSentiment_20210817.xls.xls')

df2 = preprocess_data(df2, columns=['COMMENT_TEXT', 'RATING_VALUE'],
                      label_column='RATING_VALUE', null = True,
                      labels_ok= False, dict_labels= {(' '): '', (','): '.'},
                      labels_ready=False, labels_true={(5,5): 2, (4,4):0,(1,2,3):1})

# Pré-processamento do texto

Em nosso exemplos, fizemos mais um pré-processamento para igualar a quantidade de linhas em cada sentimento, isso vai ser algo que vai variar de tabela por tabela, e eventualmente vamos ter que fazer ou deixar de fazer algum processamento nelas.

Inicialmente fazeremos a tokenização do texto depois da tokenização. Faremos a vetorização que usaremos no primeiro modelo utilizando a função `CountVectorizer` da biblioteca Sci-KitLearn.

## Tokenização

In [14]:
df_final

,COMMENT_TEXT,RATING_VALUE,sentimentos
0,aparelho básico porem muito rápido entrega per...,5.0,2
1,otimo!,5.0,2
2,Muito bom,5.0,2
3,Muito Bom,5.0,2
4,Muito bom,5.0,2
...,...,...,...
19368,Celular deveria vir limpo a cada um coloca o q...,3.0,1
19369,"Gostei muito do produto, cumpre minhas expecta...",4.0,0
19370,"Não sei se é só o meu, mas a cor da tela é est...",4.0,0
19371,Excelente produto!,4.0,0


In [15]:
portuguese_stopwords = nltk.corpus.stopwords.words('portuguese')

def preprocess_text(text, remove_stop = True, 
                    stem_words = False, remove_mentions_hashtags = True):
    """
    eg:
    input: preprocess_text("@water #dream hi hello where are you going be there tomorrow happening happen happens",  
    stem_words = True) 
    output: ['tomorrow', 'happen', 'go', 'hello']
    """

    # Remove emojis
    emoji_pattern = re.compile("[" "\U0001F1E0-\U0001F6FF" "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r"", text)
    text = "".join([x for x in text if x not in emoji.UNICODE_EMOJI])

    if remove_mentions_hashtags:
        text = re.sub(r"@(\w+)", " ", text)
        text = re.sub(r"#(\w+)", " ", text)

    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    regex = re.compile('[' + re.escape(string.punctuation) + '0-9\\r\\t\\n]')
    nopunct = regex.sub(" ", text.lower())
    words = (''.join(nopunct)).split()

    if(remove_stop):
        words = [w for w in words if w not in portuguese_stopwords]
        words = [w for w in words if len(w) > 2]  

    if(stem_words):
        stemmer = PorterStemmer()
        words = [stemmer.stem(w) for w in words]

    return list(words)

In [16]:
# cria a coluna com textos vetorizados
rows, cols = df_final.shape

df_final['token'] = [preprocess_text(df_final["COMMENT_TEXT"][row]) for row in range(rows)]

In [17]:
df_final.head()

,COMMENT_TEXT,RATING_VALUE,sentimentos,token
0,aparelho básico porem muito rápido entrega per...,5.0,2,"[aparelho, sico, porem, pido, entrega, perfeit..."
1,otimo!,5.0,2,[otimo]
2,Muito bom,5.0,2,[bom]
3,Muito Bom,5.0,2,[bom]
4,Muito bom,5.0,2,[bom]


## Vetorização

Como iremos fazer mais de um tipo de vetorização, vamos criar uma copia do dataset para utilizar nessa parte

In [ ]:
df_final1 = df_final.copy()

In [ ]:
# coleciona as palavras usadas para o treinamento da função CountVectorizer
lista_treino = []
for item in df_final1['token']:
    lista_treino1 = [n for n in item if n not in lista_treino]
    lista_treino.extend(lista_treino1)

In [ ]:
# treina o modelo de vetorização
vectorize = CountVectorizer(lowercase=True, strip_accents='unicode')

vectorize.fit(lista_treino)

In [ ]:
# criando a coluna com o texto vetorizado
def vectorize2(lista):
    lista2 = [vectorize.vocabulary_[item] for item in lista]
    
    return lista2

df_final1['vectors'] = df_final1['token'].apply(vectorize2)

#
df_final1.head()

# Machine Learning


## Organizando os variáveis.

Definiremos a variavel preditora e a variável target, depois separaremos os dados em treino e teste.

In [ ]:
#
df_final1 = shuffle(df_final1, random_state=42)

# separando as variaveis
X = df_final1['vectors'].values

#
y = df_final1['sentimentos'].values

print(X.shape, y.shape)

In [ ]:
# usa a função pad_sequences do keras para deixar todos os textos do mesmo tamanho
X = tf.keras.preprocessing.sequence.pad_sequences(X, maxlen=20)

#
X.shape

In [ ]:
# separa os dados em treino e teste e fazendo o one-hot-encoding na variável y
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.25, random_state=42)

#
ytrain = tf.keras.utils.to_categorical(ytrain)

#
ytest = tf.keras.utils.to_categorical(ytest)

#
Xtrain.shape, Xtest.shape, ytrain.shape, ytest.shape

## Criando o modelo

Vamos criar uma rede neural usando o Keras, colocaremos a primeira camada de Embedding, uma LSTM e duas camadas densas.

In [ ]:
#
model = tf.keras.Sequential()

#
model.add(tf.keras.layers.Embedding(input_dim=len(lista_treino)+1, 
                                    output_dim=128, input_shape=(Xtrain.shape[1],), 
                                    activity_regularizer=regularizers.l2(1e-5)))

#
model.add(tf.keras.layers.Dropout(0.25))

#
model.add(tf.keras.layers.LSTM(units=512, activation='relu'))

#
model.add(tf.keras.layers.Dense(units=256, activation='relu'))

#
model.add(tf.keras.layers.Dropout(0.25))

#
model.add(tf.keras.layers.Dense(units=128, activation='relu'))

#
model.add(tf.keras.layers.Dropout(0.25))

#
model.add(tf.keras.layers.Dense(units=64, activation='relu'))

#
model.add(tf.keras.layers.Dropout(0.25))

#
model.add(tf.keras.layers.Dense(units=3, activation='softmax', 
                                activity_regularizer=regularizers.l2(1e-5)))

#
model.compile(optimizer = 'Adam', 
              loss='categorical_crossentropy', 
              metrics=['accuracy']
             )

#
model.summary()

In [ ]:
# criando um stopper para a rede
stopper = EarlyStopping(monitor="val_accuracy",
                         patience=5, verbose=2, mode='max')

callbacks = [stopper]

## Treinando o modelo

Agora com o modelo criado, podemos treina-lo e analisar os resultados.

In [ ]:
history = model.fit(x=Xtrain, y=ytrain, batch_size=128,
                    validation_data=(Xtest, ytest),
                    epochs=2,callbacks=callbacks)

In [ ]:
plt.figure(figsize=[15,6])

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Acuracia Treino')
plt.plot(history.history['val_accuracy'], label='Acuracia Teste')
plt.legend()
plt.grid()

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Erro treino')
plt.plot(history.history['val_loss'], label='Erro Teste')
plt.legend()
plt.grid()



plt.show()

In [ ]:
def feelings_prediction():
    text = input("Digite seu texto aqui\n\n")
    text1 = preprocess_text(text)
    text1 = [vectorize2(text1)]
    text1 = tf.keras.preprocessing.sequence.pad_sequences(text1, maxlen=20)
    pred= model.predict(text1)
    predd = dict_feeling[np.argmax(pred)]
    print('\nSentimento analisado:', predd)

In [ ]:
feelings_prediction()

In [ ]:
feelings_prediction()

In [ ]:
feelings_prediction()

# Salvando Modelos

Para finalizar essa primeira parte, vamos salvar os modelos, tanto o modelo de vetorização treinado, quanto a rede neural

In [ ]:
dump(vectorize, 'vetorize_sklearn.joblib') 

In [ ]:
model.save('primeiro_modelo.h5')